In [0]:
%pip install pytest

In [0]:
# ============================================================
# CELL 1: INITIALIZE AUTOMATED HUMAN GATEWAY
# ============================================================

import json
from datetime import datetime, timezone

from Human_GateWay.human_gateway import HumanGateway

REVIEW_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/human_reviews"
)

gateway = HumanGateway(review_dir=REVIEW_DIR)

print("=" * 90)
print("AUTOMATED HUMAN GATEWAY INITIALIZED")
print("=" * 90)

print("Review directory:", REVIEW_DIR)
print("Gateway ready.")

In [0]:
# ============================================================
# CELL 2: IDENTIFY LATEST REVIEW PER MAPPING
# ============================================================

from datetime import datetime, timezone

EXPECTED_MAPPINGS = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

all_reviews = gateway.list_reviews()

def parse_timestamp(value):
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)

    parsed = datetime.fromisoformat(
        value.replace("Z", "+00:00")
    )

    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)

    return parsed


latest_reviews = {}

for review in all_reviews:

    mapping_name = review.get("mapping_name")

    if mapping_name not in EXPECTED_MAPPINGS:
        continue

    existing = latest_reviews.get(mapping_name)

    if (
        existing is None
        or parse_timestamp(review.get("created_at"))
        > parse_timestamp(existing.get("created_at"))
    ):
        latest_reviews[mapping_name] = review


print("=" * 100)
print("LATEST REVIEW PER MAPPING")
print("=" * 100)

for mapping_name in EXPECTED_MAPPINGS:

    review = latest_reviews.get(mapping_name)

    if not review:
        print(f"{mapping_name}: NO REVIEW FOUND")
        continue

    decision_info = review.get("decision") or {}

    decision = (
        decision_info.get("decision")
        if isinstance(decision_info, dict)
        else decision_info
    )

    print(json.dumps({
        "mapping": mapping_name,
        "review_id": review.get("review_id"),
        "status": review.get("status"),
        "decision": decision,
        "validations_passed": review.get("validations_passed"),
        "can_execute": review.get("can_execute"),
        "issues": review.get("issues", [])
    }, indent=4, default=str))


missing = set(EXPECTED_MAPPINGS) - set(latest_reviews)

if missing:
    raise ValueError(
        f"Reviews missing for mappings: {sorted(missing)}"
    )
# Compatibility alias for Cell 4
latest_reviews_by_mapping = latest_reviews

print("\nTotal mappings found:", len(latest_reviews))

In [0]:
# ============================================================
# CELL 3: VALIDATE APPROVAL ELIGIBILITY
# ============================================================

APPROVAL_ELIGIBLE_CATEGORIES = {
    "UNVERIFIED_MERGE_KEY"
}

approval_candidates = []

print("=" * 100)
print("HUMAN GATEWAY APPROVAL ELIGIBILITY")
print("=" * 100)

for mapping_name in EXPECTED_MAPPINGS:

    review = latest_reviews[mapping_name]

    review_id = review["review_id"]
    status = review.get("status")
    decision_info = review.get("decision") or {}

    decision = (
        decision_info.get("decision")
        if isinstance(decision_info, dict)
        else decision_info
    )

    issues = review.get("issues", [])

    categories = {
        str(issue.get("category", "")).strip().upper()
        for issue in issues
    }

    validations_passed = (
        review.get("validations_passed") is True
    )

    eligible = (
        status == "REVIEW_REQUIRED"
        and decision is None
        and validations_passed
        and bool(categories)
        and categories.issubset(
            APPROVAL_ELIGIBLE_CATEGORIES
        )
    )

    result = {
        "mapping": mapping_name,
        "review_id": review_id,
        "status": status,
        "decision": decision,
        "issue_categories": sorted(categories),
        "validations_passed": validations_passed,
        "eligible_for_poc_approval": eligible
    }

    approval_candidates.append(result)

    print(json.dumps(result, indent=4))


print("\nEligible reviews:",
      sum(1 for x in approval_candidates
          if x["eligible_for_poc_approval"]))

print("Not eligible:",
      sum(1 for x in approval_candidates
          if not x["eligible_for_poc_approval"]))

In [0]:

# ============================================================
# CELL 4: RECORD HUMAN GATEWAY APPROVALS
# ============================================================

import json

REVIEWER = "POC_REVIEWER"

APPROVE_MAPPINGS = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

POC_APPROVAL_COMMENT = (
    "Approved for controlled IDMC-to-Databricks POC use only. "
    "Synthetic merge keys have been reviewed and accepted as "
    "POC assumptions. This approval does not certify production "
    "business keys or authorize production execution."
)

# ------------------------------------------------------------
# 1. Validate configuration
# ------------------------------------------------------------

if not REVIEWER.strip():
    raise ValueError("Reviewer identity is required.")

if not APPROVE_MAPPINGS:
    raise ValueError("No mappings selected.")

if len(APPROVE_MAPPINGS) != len(set(APPROVE_MAPPINGS)):
    raise ValueError("Duplicate mapping names found.")

candidate_lookup = {
    item["mapping"]: item
    for item in approval_candidates
}

approval_results = []

# ------------------------------------------------------------
# 2. Process each mapping
# ------------------------------------------------------------

for mapping_name in APPROVE_MAPPINGS:

    print("\n" + "=" * 90)
    print("PROCESSING:", mapping_name)
    print("=" * 90)

    candidate = candidate_lookup.get(mapping_name)

    if not candidate:
        print("SKIPPED: Approval candidate not found.")
        approval_results.append({
            "mapping": mapping_name,
            "result": "SKIPPED",
            "reason": "CANDIDATE_NOT_FOUND"
        })
        continue

    if candidate.get("eligible_for_poc_approval") is not True:
        print("SKIPPED: Mapping is not eligible.")
        approval_results.append({
            "mapping": mapping_name,
            "result": "SKIPPED",
            "reason": "NOT_ELIGIBLE"
        })
        continue

    review_id = candidate["review_id"]

    try:
        # Fetch authoritative persisted review.
        review = gateway.get_review(review_id)

        decision_info = review.get("decision") or {}

        current_decision = (
            decision_info.get("decision")
            if isinstance(decision_info, dict)
            else decision_info
        )

        # IMPORTANT: Extract categories from the actual issues list.
        issues = review.get("issues", [])

        categories = {
            str(issue.get("category", "")).strip().upper()
            for issue in issues
            if isinstance(issue, dict)
        }

        print("Review ID:", review_id)
        print("Status:", review.get("status"))
        print("Issue categories:", sorted(categories))
        print("Validations passed:", review.get("validations_passed"))

        # Revalidate before recording.
        if review.get("status") != "REVIEW_REQUIRED":
            raise ValueError(
                f"Unexpected review status: {review.get('status')}"
            )

        if current_decision is not None:
            raise ValueError(
                f"Decision already recorded: {current_decision}"
            )

        if review.get("validations_passed") is not True:
            raise ValueError("Technical validations have not passed.")

        if categories != {"UNVERIFIED_MERGE_KEY"}:
            raise ValueError(
                f"Unexpected issue categories: {sorted(categories)}"
            )

        # Record explicit POC reviewer decision.
        gateway.record_decision_by_id(
            review_id=review_id,
            decision="APPROVE",
            reviewer=REVIEWER,
            comment=POC_APPROVAL_COMMENT
        )

        # Read back the persisted review.
        updated_review = gateway.get_review(review_id)

        updated_decision_info = updated_review.get("decision") or {}

        updated_decision = (
            updated_decision_info.get("decision")
            if isinstance(updated_decision_info, dict)
            else updated_decision_info
        )

        recorded = (
            updated_decision == "APPROVE"
            and updated_review.get("status") == "PASS"
        )

        result = {
            "mapping": mapping_name,
            "review_id": review_id,
            "decision": updated_decision,
            "status": updated_review.get("status"),
            "validations_passed": updated_review.get(
                "validations_passed"
            ),
            "can_execute": updated_review.get("can_execute"),
            "recorded": recorded
        }

        approval_results.append(result)

        print(json.dumps(result, indent=4, default=str))

    except Exception as exc:

        result = {
            "mapping": mapping_name,
            "review_id": review_id,
            "result": "FAILED",
            "error": str(exc)
        }

        approval_results.append(result)

        print(json.dumps(result, indent=4, default=str))

# ------------------------------------------------------------
# 3. Submission summary
# ------------------------------------------------------------

recorded_count = sum(
    1 for item in approval_results
    if item.get("recorded") is True
)

print("\n" + "=" * 100)
print("APPROVAL SUBMISSION SUMMARY")
print("=" * 100)

print("Total mappings:", len(APPROVE_MAPPINGS))
print("Successfully recorded:", recorded_count)
print("Failed:", len(APPROVE_MAPPINGS) - recorded_count)

for item in approval_results:
    print(json.dumps(item, indent=4, default=str))


In [0]:
# ============================================================
# CELL 5: FINAL APPROVAL VERIFICATION
# ============================================================

final_results = []

for mapping_name in APPROVE_MAPPINGS:

    candidate = next(
        item for item in approval_candidates
        if item["mapping"] == mapping_name
    )

    review_id = candidate["review_id"]

    review = gateway.get_review(review_id)

    decision_info = review.get("decision") or {}

    final_decision = (
        decision_info.get("decision")
        if isinstance(decision_info, dict)
        else decision_info
    )

    verified = (
        final_decision == "APPROVE"
        and review.get("status") == "PASS"
        and review.get("validations_passed") is True
        and review.get("can_execute") is True
        and gateway.can_proceed(
            review,
            execution_approved=False
        )
    )

    final_results.append({
        "mapping": mapping_name,
        "review_id": review_id,
        "status": review.get("status"),
        "decision": final_decision,
        "reviewer": (
            decision_info.get("reviewer")
            if isinstance(decision_info, dict)
            else None
        ),
        "verified": verified
    })


print("=" * 100)
print("HUMAN GATEWAY FINAL APPROVAL REPORT")
print("=" * 100)

for result in final_results:
    print(json.dumps(result, indent=4, default=str))


verified_count = sum(
    1 for result in final_results
    if result["verified"]
)

print("\nTotal processed:", len(final_results))
print("Verified approvals:", verified_count)
print("Not verified:", len(final_results) - verified_count)

if verified_count != len(APPROVE_MAPPINGS):
    raise RuntimeError(
        "One or more approval decisions failed verification."
    )

print("\nAll selected POC approvals successfully verified.")